> 在线讲义：[第 10 章 · 信息论：熵、交叉熵、KL 和互信息](https://wikieden.github.io/math-and-eval-stats/lessons/10-information-kl.html) · 本书《数学与实验统计》

# 第 10 章 · 信息论：熵、交叉熵、KL 和互信息

配套讲义：仓库里 `math-and-eval-stats/notes/lessons/10-information-kl.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy / SciPy（Colab 自带），不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一行代码、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 熵 | −Σp log p | `entropy([0.9, 0.1], 2)` | 0.469 比特 |
| 第二步 交叉熵 | −Σp log q | `cross_entropy(P, Q)` | 0.9856 |
| 第二步 NLL | 平均 −log q(y) | `nll_of_labels(probs, labels)` | 0.8018 |
| 第三步 KL | H(P,Q) − H(P) | `kl(P, Q)`、`kl(Q, P)` | 0.1838、0.1920 |
| 第四步 拟合 | 网格搜索 | `fit_gaussian("forward" / "reverse")` | (0, 2.06) / (−2, 0.50) |
| 第五步 互信息 | H(X)+H(Y)−H(X,Y) | `mutual_information(JOINT)` | 0.1927 |

例子都是编的：三类分布 P = [0.7, 0.2, 0.1]、Q = [0.4, 0.4, 0.2]；一个两峰分布拿单个高斯去拟合；一张 2×2 联合分布表。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import math
import numpy as np
P = np.array([0.7, 0.2, 0.1])
Q = np.array([0.4, 0.4, 0.2])
JOINT = np.array([[0.4, 0.1], [0.1, 0.4]])  # 编的：X、Y 各取 0/1 的联合概率

**函数 `entropy`**：熵 H(P) = −Σ p log p（p = 0 的项算 0）。base = 2 时单位是比特。

In [ ]:
def entropy(p, base: float = math.e) -> float:
    """熵 H(P) = −Σ p log p（p = 0 的项算 0）。base = 2 时单位是比特。"""
    p = np.asarray(p, float)
    p = p[p > 0]
    return float(-(p * np.log(p)).sum() / math.log(base))

**函数 `cross_entropy`**：交叉熵 H(P, Q) = −Σ p log q：用按 Q 设计的编码去编 P 的数据，平均要多少 nat。

In [ ]:
def cross_entropy(p, q) -> float:
    """交叉熵 H(P, Q) = −Σ p log q：用按 Q 设计的编码去编 P 的数据，平均要多少 nat。"""
    p, q = np.asarray(p, float), np.asarray(q, float)
    m = p > 0
    return float(-(p[m] * np.log(q[m])).sum())

**函数 `kl`**：KL(P‖Q) = Σ p log(p/q) = H(P, Q) − H(P) ≥ 0。

In [ ]:
def kl(p, q) -> float:
    """KL(P‖Q) = Σ p log(p/q) = H(P, Q) − H(P) ≥ 0。"""
    return cross_entropy(p, q) - entropy(p)

**函数 `nll_of_labels`**：分类器的平均负对数似然：−(1/n) Σ log q(y_i)，就是"标签的经验分布"和 q 的交叉熵。

In [ ]:
def nll_of_labels(probs: np.ndarray, labels: np.ndarray) -> float:
    """分类器的平均负对数似然：−(1/n) Σ log q(y_i)，就是"标签的经验分布"和 q 的交叉熵。"""
    return float(-np.log(probs[np.arange(len(labels)), labels]).mean())

**函数 `mixture_pdf`**：两峰：0.5·N(−2, 0.5²) + 0.5·N(2, 0.5²)。

In [ ]:
def mixture_pdf(x: np.ndarray) -> np.ndarray:
    """两峰：0.5·N(−2, 0.5²) + 0.5·N(2, 0.5²)。"""
    g = lambda m: np.exp(-0.5 * ((x - m) / 0.5) ** 2) / (0.5 * math.sqrt(2 * math.pi))
    return 0.5 * g(-2) + 0.5 * g(2)

**函数 `fit_gaussian`**：用单个高斯 N(μ, s²) 拟合两峰：forward = 最小化 KL(P‖Q)，reverse = 最小化 KL(Q‖P)。网格搜索，返回 (μ, s)。

In [ ]:
def fit_gaussian(direction: str) -> tuple[float, float]:
    """用单个高斯 N(μ, s²) 拟合两峰：forward = 最小化 KL(P‖Q)，reverse = 最小化 KL(Q‖P)。网格搜索，返回 (μ, s)。"""
    x = np.linspace(-6, 6, 2401)
    dx = x[1] - x[0]
    p = mixture_pdf(x) + 1e-300
    best = (math.inf, 0.0, 0.0)
    for mu in np.linspace(-3, 3, 121):
        for s in np.linspace(0.2, 3.0, 141):
            q = np.exp(-0.5 * ((x - mu) / s) ** 2) / (s * math.sqrt(2 * math.pi)) + 1e-300
            val = float(((p * np.log(p / q)) if direction == "forward" else (q * np.log(q / p))).sum() * dx)
            if val < best[0]:
                best = (val, float(mu), float(s))
    return best[1], best[2]

**函数 `mutual_information`**：互信息 I(X; Y) = H(X) + H(Y) − H(X, Y)。

In [ ]:
def mutual_information(joint: np.ndarray) -> float:
    """互信息 I(X; Y) = H(X) + H(Y) − H(X, Y)。"""
    return entropy(joint.sum(1)) + entropy(joint.sum(0)) - entropy(joint.ravel())

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 熵：平均有多意外")
print(f"  公平硬币 H = {entropy([0.5, 0.5], 2):.3f} 比特；9:1 的硬币 H = {entropy([0.9, 0.1], 2):.3f} 比特")
print(f"  P = {P.tolist()}：H(P) = {entropy(P):.4f} nat")

In [ ]:
print("\n第二步 交叉熵 = 负对数似然")
print(f"  H(P, Q) = {cross_entropy(P, Q):.4f} nat（比 H(P) 多出来的就是 KL）")
probs = np.array([[0.7, 0.2, 0.1]] * 10)
labels = np.array([0] * 7 + [1] * 2 + [2])  # 10 个样本，标签的经验分布正好是 P
print(f"  10 个样本都预测 [0.7, 0.2, 0.1]，标签 7/2/1 个：平均 NLL = {nll_of_labels(probs, labels):.4f} = H(P)")

In [ ]:
print("\n第三步 KL 不对称")
print(f"  KL(P‖Q) = {kl(P, Q):.4f}，KL(Q‖P) = {kl(Q, P):.4f}")

In [ ]:
print("\n第四步 正向 KL 盖住两峰，反向 KL 挑一个峰")
for d in ("forward", "reverse"):
    mu, s = fit_gaussian(d)
    print(f"  {'KL(P‖Q) 正向' if d == 'forward' else 'KL(Q‖P) 反向'}：μ = {mu:+.2f}，s = {s:.2f}")

In [ ]:
print("\n第五步 互信息：知道 Y 以后，X 的不确定性少了多少")
print(f"  联合表 {JOINT.tolist()}：I(X; Y) = {mutual_information(JOINT):.4f} nat = {mutual_information(JOINT) / math.log(2):.4f} 比特")

## 3. 代码里才看得到的

1. **0·log 0 = 0**：直接算会得到 nan，要先把 p = 0 的项去掉。
2. **连续分布的 KL 用格点近似**：在 [−6, 6] 上取 2401 个点，求和乘格宽；两头加 1e-300 防止 log 0。
3. **反向 KL 的最优解有两个**（μ = −2 或 +2，对称）：网格搜索从左往右找，先碰到的是 −2。

## 4. 自己改一改

1. 把两峰的间距从 4 改成 1（两峰重叠），正向和反向 KL 的拟合结果还差那么多吗？
2. 把 `JOINT` 改成 [[0.25, 0.25], [0.25, 0.25]]，互信息是多少？

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 熵是平均意外，越平越大越尖越小；
> 交叉熵用错的地图走路，就是负对数似然；
> KL 是多走的冤枉路，不对称不是距离；
> 正向盖全、反向挑峰；互信息是重叠的那一块。